In [2]:
# !pip install scikit-learn pandas datasets
import re, io, json, zipfile, urllib.request
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, precision_recall_fscore_support

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

In [4]:
!pip install -q "datasets==3.6.0"

In [2]:
import numpy, pyarrow, datasets
print(numpy.__version__, pyarrow.__version__, datasets.__version__)

1.26.4 18.1.0 3.6.0


In [4]:
import os, io, json, zipfile, urllib.request
from pathlib import Path
from collections import Counter
import pandas as pd

DATA = Path.home() / "lease-checker" / "data" / "external"
DATA.mkdir(parents=True, exist_ok=True)

def download_zip(url, dest):
    """Download a zip and extract it into dest (skips if already there)."""
    dest = Path(dest)
    if dest.exists() and any(dest.iterdir()):
        print("already downloaded:", dest); return dest
    print("downloading", url, "...")
    with urllib.request.urlopen(url) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(dest)
    print("saved to", dest); return dest

lease_dir = download_zip(
    "https://gitlab.com/spyretta.leiv/lease_contract_review/-/archive/master/lease_contract_review-master.zip",
    DATA / "lease_contract_review")

files = [p for p in lease_dir.rglob("*") if p.is_file()]
print(len(files), "files;", Counter(p.suffix for p in files).most_common())
for p in files[:30]:
    print("  ", p.relative_to(lease_dir), f"({p.stat().st_size // 1024} KB)")

downloading https://gitlab.com/spyretta.leiv/lease_contract_review/-/archive/master/lease_contract_review-master.zip ...
saved to /Users/marufi/lease-checker/data/external/lease_contract_review
88 files; [('.py', 27), ('.csv', 17), ('.pkl', 9), ('.json', 7), ('.pdf', 7), ('.txt', 6), ('.ipynb', 5), ('', 4), ('.md', 3), ('.model', 1), ('.bin', 1), ('.zip', 1)]
   lease_contract_review-master/README.md (0 KB)
   lease_contract_review-master/resources/.DS_Store (6 KB)
   lease_contract_review-master/resources/ALeaseBert/config.json (0 KB)
   lease_contract_review-master/resources/ALeaseBert/spiece.model (742 KB)
   lease_contract_review-master/resources/ALeaseBert/pytorch_model.bin (46149 KB)
   lease_contract_review-master/resources/source_code/.DS_Store (6 KB)
   lease_contract_review-master/resources/source_code/requirements.txt (2 KB)
   lease_contract_review-master/resources/source_code/README.md (0 KB)
   lease_contract_review-master/resources/source_code/.gitignore (0 KB)
   lease_

In [6]:
seen = set()
for p in files:
    if p.suffix in seen or p.suffix in {".png", ".jpg", ".pdf", ".zip", ".bin", ".pt"}:
        continue
    seen.add(p.suffix)
    print("=====", p.relative_to(lease_dir))
    print(p.read_text(errors="ignore")[:800], "\n")

===== lease_contract_review-master/README.md
# lease_contract_review

work in progress...
 

===== lease_contract_review-master/resources/.DS_Store
   Bud1                                                                    s e B e r t                                          
 A L e a s e B e r tdsclbool     s o u r c e _ c o d edsclbool                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

===== lease_contract_review-master/resources/ALeaseBert

In [8]:
# 1) How many files are in each folder?
folder_counts = Counter(str(p.parent.relative_to(lease_dir)) for p in files)
for folder, n in sorted(folder_counts.items()):
    print(f"{n:5d}  {folder}")

# 2) List every file that might be data (skip model weights)
DATA_EXT = {".csv", ".tsv", ".json", ".jsonl", ".txt", ".xlsx", ".xls", ".pkl", ".parquet", ".conll", ".xml", ".py", ".ipynb"}
print("\nPossible data / code files:")
for p in files:
    if p.suffix.lower() in DATA_EXT and "ALeaseBert" not in str(p):
        print(f"   {p.relative_to(lease_dir)}  ({p.stat().st_size // 1024} KB)")

    1  lease_contract_review-master
    1  lease_contract_review-master/resources
    3  lease_contract_review-master/resources/ALeaseBert
    4  lease_contract_review-master/resources/source_code
   12  lease_contract_review-master/resources/source_code/data/consolidated
    4  lease_contract_review-master/resources/source_code/data/consolidated/ner
    2  lease_contract_review-master/resources/source_code/data/original
   16  lease_contract_review-master/resources/source_code/data/paper
    1  lease_contract_review-master/resources/source_code/data/paper/.ipynb_checkpoints
    1  lease_contract_review-master/resources/source_code/eval
    1  lease_contract_review-master/resources/source_code/eval/.ipynb_checkpoints
    4  lease_contract_review-master/resources/source_code/eval/Albert_AddPT
    4  lease_contract_review-master/resources/source_code/eval/Albert_Base
    4  lease_contract_review-master/resources/source_code/eval/CRF
   20  lease_contract_review-master/resources/source_co

In [10]:
LCR = lease_dir / "lease_contract_review-master/resources/source_code/data/consolidated"

for name in ["rf_train.csv", "rf_val.csv", "easy_rf_train.csv", "cl_train.csv", "cl_val.csv"]:
    d = pd.read_csv(LCR / name)
    print(f"===== {name}   shape={d.shape}")
    print("columns:", list(d.columns))
    print(d.head(3).to_string(max_colwidth=120))
    # show label distribution for any short-valued column
    for col in d.columns:
        if d[col].nunique() <= 30:
            print(f"\n  value counts of '{col}':")
            print(d[col].value_counts().head(30).to_string())
    print("\n")

print("===== annotations-legend.json")
print((LCR.parent / "original/annotations-legend.json").read_text())

===== rf_train.csv   shape=(42282, 7)
columns: ['uuid', 'text', 'type', 'raw_text', 'start', 'end', 'binary']
                                     uuid text  type                                                                                                                 raw_text  start  end  binary
0  aL89QqQu924hlLlvJhHT7QUdXkDe-lease_579  NaN  none                                   receipt and sufficiency of which are hereby acknowledged, has remised, released, sold,      0    0       0
1  ak5X4K4XyNWp3lmPt3oZ1YnTRs3m-lease_505  NaN  none                                                                                                      THE SECOND SCHEDULE      0    0       0
2  aau3VbTW0f0FXCd8JUWypeJZBfLy-lease_764  NaN  none  14. (a) The Bank and, as applicable, Bancorp shall submit written plans and programs that are acceptable to the Rese...      0    0       0

  value counts of 'type':
type
none                                       41592
services_charges                 

In [12]:
def load_lcr_rf(name, split):
    d = pd.read_csv(LCR / name)
    d = d.rename(columns={"raw_text": "text_raw"})
    d["text"] = d["text_raw"].astype(str).str.split().str.join(" ")
    d = d[d["text"].str.len() >= 40]                    # drop headings / fragments
    d = d.drop_duplicates("text")
    d["split"] = split
    return d[["uuid", "text", "type", "binary", "split"]]

lcr = pd.concat([load_lcr_rf("rf_train.csv", "train"),
                 load_lcr_rf("rf_val.csv", "val")], ignore_index=True)
print("after cleaning:", lcr.shape)
print(lcr.groupby("split")["binary"].agg(["count", "sum"]))

# 3 real examples of each red-flag type, so we can decide what counts as "risky for a tenant"
pd.set_option("display.max_colwidth", 250)
for t, g in lcr[lcr.binary == 1].groupby("type"):
    print(f"\n===== {t}  (n={len(g)})")
    for s in g["text"].sample(min(3, len(g)), random_state=42):
        print("  •", s[:250])

lcr.to_csv(DATA / "lcr_redflags_clean.csv", index=False)
print("\nsaved:", DATA / "lcr_redflags_clean.csv")

after cleaning: (23450, 5)
       count  sum
split            
train  18444  581
val     5006  163

===== additional_remarks  (n=2)
  • 4.2.3 Party B shall manage the fruit trees in the orchard carefully during the contracting period. Party B shall purchase seedlings and replant them if there is any damage. If there is any damage to the orange trees due to mismanagement (such as the 
  • 4.6 Within five (5) business days from the Commencement Date, the Tenant shall deliver the Landlord Ten Thousand (10,000) Shares of its Common Stock as additional consideration for entering into this Lease.

===== assignment_indeplaatsstelling_permitted  (n=1)
  • 1.During the term of this contract, Party A shall have the right to transfer some or all of its rights and obligations to a third party, but such transfer shall not affect Party B’s and Party C’s ability to regain ownership of the Leased Items upon e

===== break_option  (n=95)
  • 8.1 Where the land of the plant has to be removed as requeste

In [14]:
LCR_TO_TYPE = {
    "landlord_repairs": "repairs",
    "services_charges": "fees",
    "guarantee_transferable": "deposit",
    "warrantees_of_the_owner": "deposit",
    "break_option": "termination",
    "reinstatement_clause": "termination",
    "extension_period": "termination",
    "right_of_first_refusal_to_lease": "termination",
    "damage": "liability",
}

# (a) clause-type examples from real leases
lcr_types = lcr[lcr["type"].isin(LCR_TO_TYPE)].copy()
lcr_types["clause_type"] = lcr_types["type"].map(LCR_TO_TYPE)
lcr_types["source"] = "lease_contract_review"
print("clause-type examples:", len(lcr_types))
print(lcr_types["clause_type"].value_counts().to_string())

# (b) false-alarm test set: real lease sentences with no tag (assumed normal), 2,000 sampled
lcr_normal = (lcr[(lcr["type"] == "none") & (lcr["split"] == "val")]
              .sample(2000, random_state=42).copy())
lcr_normal["risky"] = 0
lcr_normal["source"] = "lease_contract_review"
print("\nfalse-alarm test sentences:", len(lcr_normal))

lcr_types[["text", "clause_type", "source"]].to_csv(DATA / "lcr_clause_types.csv", index=False)
lcr_normal[["text", "risky", "source"]].to_csv(DATA / "lcr_normal_test.csv", index=False)
print("saved both to", DATA)

clause-type examples: 551
clause_type
termination    224
fees           135
repairs         69
deposit         69
liability       54

false-alarm test sentences: 2000
saved both to /Users/marufi/lease-checker/data/external


In [16]:
from datasets import load_dataset

unfair = load_dataset("coastalcph/lex_glue", "unfair_tos")
UNFAIR_NAMES = unfair["train"].features["labels"].feature.names
print(unfair)
print("label names:", UNFAIR_NAMES)

def unfair_to_df(split):
    d = unfair[split].to_pandas()
    d["risky"] = d["labels"].apply(lambda l: int(len(l) > 0))
    d["unfair_types"] = d["labels"].apply(lambda l: [UNFAIR_NAMES[i] for i in l])
    d["split"] = split
    return d[["text", "risky", "unfair_types", "split"]]

ut = pd.concat([unfair_to_df(s) for s in ["train", "validation", "test"]], ignore_index=True)
print(ut.groupby("split")["risky"].agg(["count", "sum"]))

# 2 examples of each unfair type
for name in UNFAIR_NAMES:
    ex = ut[ut["unfair_types"].apply(lambda l: name in l)]["text"]
    print(f"\n===== {name}  (n={len(ex)})")
    for s in ex.sample(min(2, len(ex)), random_state=42):
        print("  •", s[:220])

ut.to_csv(DATA / "unfair_tos.csv", index=False)

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

unfair_tos/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  501kB            

unfair_tos/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

unfair_tos/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  147kB            

unfair_tos/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

unfair_tos/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  218kB            

unfair_tos/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5532 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1607 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2275 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 5532
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 1607
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 2275
    })
})
label names: ['Limitation of liability', 'Unilateral termination', 'Unilateral change', 'Content removal', 'Contract by using', 'Choice of law', 'Jurisdiction', 'Arbitration']
            count  sum
split                 
test         1607  172
train        5532  630
validation   2275  230

===== Limitation of liability  (n=296)
  • in no event shall the liability of linkedin and its affiliates ( and those that linkedin works with to provide the services ) exceed , in the aggregate for all claims , an amount that is the lesser of ( a ) five times th
  • to the fullest extent permitted by law , in no event will spotify , its officers , shareholders , employees , agents , directors , subsidiaries , affili

In [18]:
def detok(t):
    t = re.sub(r"\s+([,.;:!?)\]'])", r"\1", t)     # "word ," -> "word,"
    t = re.sub(r"([(\[])\s+", r"\1", t)            # "( a" -> "(a"
    t = t.replace("``", '"').replace("''", '"')
    t = re.sub(r"\s+'s\b", "'s", t)
    return re.sub(r"\s+", " ", t).strip()

import re
KEEP = {"Limitation of liability": "liability",
        "Unilateral termination": "termination",
        "Unilateral change": None,
        "Arbitration": "liability",
        "Jurisdiction": "liability"}
DROP = {"Content removal", "Contract by using", "Choice of law"}

ut["text"] = ut["text"].apply(detok)
ut = ut[ut["text"].str.len() >= 40].drop_duplicates("text")

def map_row(types):
    if any(t in DROP for t in types) and not any(t in KEEP for t in types):
        return "DROP"
    kept = [KEEP[t] for t in types if t in KEEP and KEEP[t]]
    return kept[0] if kept else None

ut["clause_type"] = ut["unfair_types"].apply(map_row)
ut = ut[ut["clause_type"] != "DROP"].copy()

# keep all useful unfair sentences + 2x as many fair sentences (keeps balance reasonable)
ut_risky = ut[ut["risky"] == 1]
ut_fair = ut[ut["risky"] == 0].sample(2 * len(ut_risky), random_state=42)
ut_use = pd.concat([ut_risky, ut_fair], ignore_index=True)
ut_use["source"] = "unfair_tos"

print(ut_use.groupby(["split", "risky"]).size().unstack())
print("\nclause types among risky:\n", ut_risky["clause_type"].value_counts(dropna=False).to_string())
print("\nexample after cleaning:\n ", ut_risky["text"].iloc[0][:200])

ut_use[["text", "risky", "clause_type", "split", "source"]].to_csv(DATA / "unfair_tos_clean.csv", index=False)

risky         0    1
split               
test        241  127
train       901  468
validation  396  174

clause types among risky:
 clause_type
liability      403
termination    220
None           146

example after cleaning:
  we may make changes to this agreement and to the services from time to time.


In [20]:
ledgar = load_dataset("coastalcph/lex_glue", "ledgar")
LEDGAR_NAMES = ledgar["train"].features["label"].names
print(ledgar)

lg = pd.concat([ledgar[s].to_pandas().assign(split=s) for s in ["train", "validation", "test"]],
               ignore_index=True)
lg["label_name"] = lg["label"].map(lambda i: LEDGAR_NAMES[i])
print("\nall 100 labels with counts:")
print(lg["label_name"].value_counts().to_string())

lg.to_csv(DATA / "ledgar.csv", index=False)

ledgar/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.9MB            

ledgar/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.31MB            

ledgar/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.44MB            

ledgar/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
})

all 100 labels with counts:
label_name
Governing Laws                  4243
Counterparts                    3346
Notices                         3313
Entire Agreements               3105
Severability                    2552
Survival                        1951
Amendments                      1948
Assignments                     1730
Expenses                        1577
Terms                           1511
Taxes                           1488
Insurances                      1441
Terminations                    1423
Compliance With Laws            1335
Litigations                     1329
General                         1329
Confidentiality                 1328
Further Assurances              1316
Indemn

In [22]:
LEDGAR_TO_TYPE = {
    "Terminations": "termination",
    "Fees": "fees", "Expenses": "fees", "Costs": "fees", "Payments": "fees",
    "Insurances": "liability", "Indemnifications": "liability", "Indemnity": "liability",
    "Releases": "liability", "Remedies": "liability",
}
BOILERPLATE = ["Counterparts", "Headings", "Severability", "Entire Agreements",
               "Notices", "Effective Dates"]

lg["text"] = lg["text"].astype(str).str.split().str.join(" ")
lg = lg[lg["text"].str.len().between(40, 1500)].drop_duplicates("text")

# (a) clause-type examples, max 300 per type
lg_types = lg[lg["label_name"].isin(LEDGAR_TO_TYPE)].copy()
lg_types["clause_type"] = lg_types["label_name"].map(LEDGAR_TO_TYPE)
lg_types = (lg_types.groupby("clause_type", group_keys=False)
            .apply(lambda g: g.sample(min(300, len(g)), random_state=42)))
lg_types["source"] = "ledgar"

# (b) boilerplate = normal examples for the risk model, 100 per label
lg_normal = (lg[lg["label_name"].isin(BOILERPLATE)]
             .groupby("label_name", group_keys=False)
             .apply(lambda g: g.sample(min(100, len(g)), random_state=42)).copy())
lg_normal["risky"] = 0
lg_normal["source"] = "ledgar"

print("clause-type examples:\n", lg_types["clause_type"].value_counts().to_string())
print("\nboilerplate normal examples:", len(lg_normal))
print("\nexample:", lg_types["text"].iloc[0][:200])

lg_types[["text", "clause_type", "split", "source"]].to_csv(DATA / "ledgar_clause_types.csv", index=False)
lg_normal[["text", "risky", "split", "source"]].to_csv(DATA / "ledgar_normal.csv", index=False)

clause-type examples:
 clause_type
fees           300
liability      300
termination    300

boilerplate normal examples: 600

example: (a) The Borrowers agree to pay to the Administrative Agent, in US Dollars, for the account of each Lender, a commitment fee (a “ Commitment Fee ”), which shall accrue at the Applicable Rate on the dai


/var/folders/6_/_xfstgh17xq18qprztkq4t_00000gn/T/ipykernel_36134/2226566210.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(300, len(g)), random_state=42)))
/var/folders/6_/_xfstgh17xq18qprztkq4t_00000gn/T/ipykernel_36134/2226566210.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(100, len(g)), random_state=42)).copy())


In [24]:
cuad_dir = download_zip("https://github.com/TheAtticusProject/cuad/raw/main/data.zip", DATA / "cuad")
print([p.name for p in cuad_dir.rglob("*") if p.is_file()][:20])

cuad_json = next(cuad_dir.rglob("CUADv1.json"))

def load_cuad(json_path):
    rows = []
    for contract in json.load(open(json_path, encoding="utf-8"))["data"]:
        for para in contract["paragraphs"]:
            for qa in para["qas"]:
                category = qa["id"].split("__")[-1]
                for ans in qa["answers"]:
                    rows.append((contract["title"], category, ans["text"]))
    return pd.DataFrame(rows, columns=["contract", "category", "clause"]).drop_duplicates()

cuad_df = load_cuad(cuad_json)
print(cuad_df.shape, "| contracts:", cuad_df.contract.nunique(), "| categories:", cuad_df.category.nunique())
print(cuad_df["category"].value_counts().to_string())

# 2 examples for the categories most likely to matter for tenants
for cat in ["Uncapped Liability", "Cap On Liability", "Termination For Convenience",
            "Liquidated Damages", "Renewal Term", "Notice Period To Terminate Renewal",
            "Covenant Not To Sue", "Insurance"]:
    ex = cuad_df[cuad_df["category"] == cat]["clause"]
    print(f"\n===== {cat}  (n={len(ex)})")
    for s in ex.sample(min(2, len(ex)), random_state=42):
        print("  •", " ".join(s.split())[:220])

cuad_df.to_csv(DATA / "cuad_clauses.csv", index=False)

downloading https://github.com/TheAtticusProject/cuad/raw/main/data.zip ...
saved to /Users/marufi/lease-checker/data/external/cuad
['CUADv1.json', 'test.json', 'train_separate_questions.json']
(13823, 3) | contracts: 510 | categories: 41
category
Parties                               2554
License Grant                          777
Cap On Liability                       672
Anti-Assignment                        654
Audit Rights                           643
Insurance                              560
Document Name                          521
Agreement Date                         476
Expiration Date                        467
Governing Law                          464
Post-Termination Services              450
Effective Date                         447
Minimum Commitment                     424
Revenue/Profit Sharing                 418
Exclusivity                            410
Rofr/Rofo/Rofn                         367
Ip Ownership Assignment                318
Non-Transferable Lice

In [26]:
CUAD_TO_TYPE = {
    "Termination For Convenience": "termination", "Renewal Term": "termination",
    "Notice Period To Terminate Renewal": "termination", "Expiration Date": "termination",
    "Cap On Liability": "liability", "Uncapped Liability": "liability",
    "Insurance": "liability", "Covenant Not To Sue": "liability",
    "Liquidated Damages": "fees",
}
cu = cuad_df.copy()
cu["text"] = cu["clause"].astype(str).str.split().str.join(" ")
cu = cu[cu["text"].str.len().between(40, 1500)].drop_duplicates("text")
cu = cu[cu["category"].isin(CUAD_TO_TYPE)].copy()
cu["clause_type"] = cu["category"].map(CUAD_TO_TYPE)
cu = (cu.groupby("clause_type", group_keys=False)
        .apply(lambda g: g.sample(min(300, len(g)), random_state=42)))
cu["source"] = "cuad"
# split by contract so the same contract never appears in both train and test
test_contracts = set(pd.Series(cu["contract"].unique()).sample(frac=0.2, random_state=42))
cu["split"] = cu["contract"].apply(lambda c: "test" if c in test_contracts else "train")

print(cu.groupby(["clause_type", "split"]).size().unstack().to_string())
cu[["text", "clause_type", "split", "source"]].to_csv(DATA / "cuad_clause_types.csv", index=False)

split        test  train
clause_type             
fees           22     80
liability      59    241
termination    61    239


/var/folders/6_/_xfstgh17xq18qprztkq4t_00000gn/T/ipykernel_36134/1373124231.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(300, len(g)), random_state=42)))


In [32]:
PROJECT = Path.home() / "lease-checker"

def norm(t):
    t = re.sub(r"^\s*(?:\(?[a-z0-9]{1,3}[\.\)]\s*)+", "", str(t))   # strip "1.", "(a)", "4.2.3"
    return re.sub(r"\s+", " ", t).strip()

# ---------- RISK table ----------
own = pd.read_csv(PROJECT / "data" / "dataset.csv")
own = pd.DataFrame({"text": own["clause"], "risky": own["label"], "source": "own", "split": "train"})

ut_r = pd.read_csv(DATA / "unfair_tos_clean.csv")
ut_r["split"] = ut_r["split"].replace({"validation": "train"})
lg_n = pd.read_csv(DATA / "ledgar_normal.csv")
lg_n["split"] = lg_n["split"].replace({"validation": "train"})
lcr_n = pd.read_csv(DATA / "lcr_normal_test.csv").assign(split="false_alarm_test")

risk_all = pd.concat([own, ut_r[["text", "risky", "source", "split"]],
                      lg_n[["text", "risky", "source", "split"]],
                      lcr_n[["text", "risky", "source", "split"]]], ignore_index=True)
risk_all["text"] = risk_all["text"].apply(norm)
risk_all = risk_all[risk_all["text"].str.len() >= 30].drop_duplicates("text")

# ---------- TYPE table ----------
seed = pd.DataFrame(SEED, columns=["text", "clause_type", "risky"])
seed = seed.assign(source="seed", split="train")[["text", "clause_type", "source", "split"]]

lcr_t = pd.read_csv(DATA / "lcr_clause_types.csv")
lcr_t["split"] = "train"
lg_t = pd.read_csv(DATA / "ledgar_clause_types.csv")
lg_t["split"] = lg_t["split"].replace({"validation": "train"})
cu_t = pd.read_csv(DATA / "cuad_clause_types.csv")

type_all = pd.concat([seed, lcr_t, lg_t, cu_t], ignore_index=True)
type_all["text"] = type_all["text"].apply(norm)
type_all = type_all[type_all["text"].str.len() >= 30].drop_duplicates("text")

# ---------- summary ----------
print("=== RISK table ===")
print(risk_all.groupby(["source", "split", "risky"]).size().unstack(fill_value=0).to_string())
print("\n=== TYPE table ===")
print(type_all.groupby(["clause_type", "source"]).size().unstack(fill_value=0).to_string())

risk_all.to_csv(DATA / "combined_risk.csv", index=False)
type_all.to_csv(DATA / "combined_types.csv", index=False)
print("\nsaved combined_risk.csv and combined_types.csv")

=== RISK table ===
risky                                      0    1
source                split                      
lease_contract_review false_alarm_test  1999    0
ledgar                test                89    0
                      train              511    0
own                   train              131   32
unfair_tos            test               241  127
                      train             1295  641

=== TYPE table ===
source       cuad  lease_contract_review  ledgar  seed
clause_type                                           
deposit         0                     69       0     8
entry           0                      0       0     8
fees          102                    134     300     8
liability     300                     54     300     8
pets_guests     0                      0       0     8
rent            0                      0       0     8
repairs         0                     68       0     8
termination   300                    219     300     8

saved comb

In [30]:
import ast

src = (Path.home() / "Downloads" / "lease_checker_all_cells.py").read_text()
tree = ast.parse(src)
SEED = next(ast.literal_eval(node.value) for node in tree.body
            if isinstance(node, ast.Assign) and getattr(node.targets[0], "id", "") == "SEED")
print(len(SEED), "seed clauses loaded")

pd.DataFrame(SEED, columns=["text", "clause_type", "risky"]).to_csv(
    PROJECT / "data" / "seed_clauses.csv", index=False)
print("saved to", PROJECT / "data" / "seed_clauses.csv")

64 seed clauses loaded
saved to /Users/marufi/lease-checker/data/seed_clauses.csv


In [34]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import joblib, json, numpy as np

encoder = SentenceTransformer("all-MiniLM-L6-v2")
def embed(texts):
    return encoder.encode(list(texts), normalize_embeddings=True, batch_size=64, show_progress_bar=True)

# same prefix-stripping as lease_checker.py, so old and new models see identical input
PREFIX = r"^\s*(?:\d{1,2}(?:\.\d+)*[\.\)]?\s+)?(?:[A-Z][A-Z\s/&'\-]{2,}[:\.]\s*)?"
def strip_prefix(t): return re.sub(PREFIX, "", str(t)).strip() or str(t)

RULES = json.load(open(PROJECT / "lease_rules.json"))
def rule_flag(t): return int(any(re.search(pat, t.lower()) for _, pat, _ in RULES))

# ---------- train ----------
risk_all = pd.read_csv(DATA / "combined_risk.csv")
train = risk_all[risk_all["split"] == "train"]
X_train = embed(train["text"].apply(strip_prefix))
new_model = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
new_model.fit(X_train, train["risky"].values)
old_model = joblib.load(PROJECT / "lease_emb_model.joblib")

# ---------- test sets ----------
final = pd.read_csv(PROJECT / "final_test_results.csv")
tests = {
    "Your final test (20)":        pd.DataFrame({"text": final["clause"], "risky": final["expected"]}),
    "UNFAIR-ToS test":             risk_all[(risk_all.source == "unfair_tos") & (risk_all.split == "test")],
    "LEDGAR boilerplate test":     risk_all[(risk_all.source == "ledgar") & (risk_all.split == "test")],
    "Real-lease false-alarm test": risk_all[risk_all.split == "false_alarm_test"],
}

rows, preds = [], {}
for tname, d in tests.items():
    texts = d["text"].astype(str).tolist()
    X = embed([strip_prefix(t) for t in texts])
    rules = np.array([rule_flag(t) for t in texts])
    y = d["risky"].values
    for mname, m in [("old", old_model), ("new", new_model)]:
        ml = (m.predict_proba(X)[:, 1] >= 0.5).astype(int)
        for method, p in [("model only", ml), ("model OR rules", ml | rules)]:
            preds[(tname, mname, method)] = p
            if y.sum() > 0:
                pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
                rows.append((tname, mname, method, accuracy_score(y, p), pr, rc, f1, np.nan))
            else:
                rows.append((tname, mname, method, accuracy_score(y, p), np.nan, np.nan, np.nan, p.mean()))

results = pd.DataFrame(rows, columns=["test set", "model", "method", "accuracy",
                                      "precision", "recall", "F1", "false alarm rate"]).round(2)
print(results.to_string(index=False))

# ---------- what does the new system still get wrong on your 20? ----------
p = preds[("Your final test (20)", "new", "model OR rules")]
print("\nErrors on your final test (new model OR rules):")
for t, y_true, y_pred in zip(final["clause"], final["expected"], p):
    if y_true != y_pred:
        print(f"  expected {y_true}, got {y_pred}:  {t}")

joblib.dump(new_model, PROJECT / "lease_emb_model_v2.joblib")
print("\nsaved lease_emb_model_v2.joblib (old model untouched)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

                   test set model         method  accuracy  precision  recall   F1  false alarm rate
       Your final test (20)   old     model only      0.65       0.71    0.50 0.59               NaN
       Your final test (20)   old model OR rules      0.65       0.71    0.50 0.59               NaN
       Your final test (20)   new     model only      0.55       0.57    0.40 0.47               NaN
       Your final test (20)   new model OR rules      0.55       0.57    0.40 0.47               NaN
            UNFAIR-ToS test   old     model only      0.65       0.50    0.13 0.20               NaN
            UNFAIR-ToS test   old model OR rules      0.67       0.57    0.20 0.30               NaN
            UNFAIR-ToS test   new     model only      0.81       0.70    0.80 0.75               NaN
            UNFAIR-ToS test   new model OR rules      0.80       0.68    0.81 0.74               NaN
    LEDGAR boilerplate test   old     model only      0.99        NaN     NaN  NaN         

In [36]:
from sklearn.model_selection import StratifiedKFold

lcr_all = pd.read_csv(DATA / "lcr_redflags_clean.csv")
pool = lcr_all[(lcr_all.split == "train") & (lcr_all.type == "none")].sample(1500, random_state=1)
lcr_add, lcr_valfa = pool.iloc[:500], pool.iloc[500:]          # 500 for training, 1000 for choosing
X_add   = embed(lcr_add["text"].apply(norm).apply(strip_prefix))
X_valfa = embed(lcr_valfa["text"].apply(norm).apply(strip_prefix))   # NOT the false-alarm test set

y_tr   = train["risky"].values
is_own = (train["source"] == "own").values
own_idx, other_idx = np.where(is_own)[0], np.where(~is_own)[0]

def fit(idx, w_own, add):
    X, y = X_train[idx], y_tr[idx]
    w = np.where(is_own[idx], w_own, 1.0)
    if add:
        X = np.vstack([X, X_add]); y = np.concatenate([y, np.zeros(len(X_add), int)])
        w = np.concatenate([w, np.ones(len(X_add))])
    return LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42).fit(X, y, sample_weight=w)

skf = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for add in [False, True]:
    for w_own in [1, 5, 10, 20]:
        # 5-fold CV on YOUR clauses: each fold of your data is held out, everything else trains
        oof = np.zeros(len(own_idx))
        for tr, te in skf.split(own_idx, y_tr[own_idx]):
            m = fit(np.concatenate([other_idx, own_idx[tr]]), w_own, add)
            oof[te] = m.predict_proba(X_train[own_idx[te]])[:, 1]
        pr, rc, f1, _ = precision_recall_fscore_support(y_tr[own_idx], (oof >= 0.5).astype(int),
                                                        average="binary", zero_division=0)
        fa = (fit(np.arange(len(y_tr)), w_own, add).predict_proba(X_valfa)[:, 1] >= 0.5).mean()
        rows.append((add, w_own, pr, rc, f1, fa))

grid = pd.DataFrame(rows, columns=["+500 lease normals", "own weight",
                                   "own CV precision", "own CV recall", "own CV F1", "false alarm (val)"]).round(3)
print(grid.to_string(index=False))

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

 +500 lease normals  own weight  own CV precision  own CV recall  own CV F1  false alarm (val)
              False           1             0.333          0.500      0.400              0.151
              False           5             0.409          0.562      0.474              0.121
              False          10             0.410          0.500      0.451              0.112
              False          20             0.417          0.469      0.441              0.112
               True           1             0.393          0.344      0.367              0.050
               True           5             0.366          0.469      0.411              0.053
               True          10             0.395          0.469      0.429              0.056
               True          20             0.417          0.469      0.441              0.062


In [38]:
W_OWN, ADD = 10, True
final_model = fit(np.arange(len(y_tr)), W_OWN, ADD)

rows, preds = [], {}
for tname, d in tests.items():
    texts = d["text"].astype(str).tolist()
    X = embed([strip_prefix(t) for t in texts])
    rules = np.array([rule_flag(t) for t in texts])
    y = d["risky"].values
    for mname, m in [("old", old_model), ("new v1 (unweighted)", new_model), ("new v2 (chosen)", final_model)]:
        ml = (m.predict_proba(X)[:, 1] >= 0.5).astype(int)
        p = ml | rules                                   # the real system = model OR rules
        preds[(tname, mname)] = p
        if y.sum() > 0:
            pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
            rows.append((tname, mname, accuracy_score(y, p), pr, rc, f1, np.nan))
        else:
            rows.append((tname, mname, accuracy_score(y, p), np.nan, np.nan, np.nan, p.mean()))

final_results = pd.DataFrame(rows, columns=["test set", "model", "accuracy", "precision",
                                            "recall", "F1", "false alarm rate"]).round(2)
print("All rows = model OR rules (the full system)\n")
print(final_results.to_string(index=False))

p = preds[("Your final test (20)", "new v2 (chosen)")]
print("\nErrors on your final test (new v2):")
for t, y_true, y_pred in zip(final["clause"], final["expected"], p):
    if y_true != y_pred:
        print(f"  expected {y_true}, got {y_pred}:  {t}")

joblib.dump(final_model, PROJECT / "lease_emb_model_v2.joblib")
final_results.to_csv(PROJECT / "final_results_v2.csv", index=False)
print("\nsaved lease_emb_model_v2.joblib and final_results_v2.csv")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

All rows = model OR rules (the full system)

                   test set               model  accuracy  precision  recall   F1  false alarm rate
       Your final test (20)                 old      0.65       0.71    0.50 0.59               NaN
       Your final test (20) new v1 (unweighted)      0.55       0.57    0.40 0.47               NaN
       Your final test (20)     new v2 (chosen)      0.55       0.57    0.40 0.47               NaN
            UNFAIR-ToS test                 old      0.67       0.57    0.20 0.30               NaN
            UNFAIR-ToS test new v1 (unweighted)      0.80       0.68    0.81 0.74               NaN
            UNFAIR-ToS test     new v2 (chosen)      0.77       0.63    0.82 0.71               NaN
    LEDGAR boilerplate test                 old      0.99        NaN     NaN  NaN              0.01
    LEDGAR boilerplate test new v1 (unweighted)      0.99        NaN     NaN  NaN              0.01
    LEDGAR boilerplate test     new v2 (chosen)      0.

In [40]:
X20 = embed([strip_prefix(t) for t in final["clause"]])
prob20 = final_model.predict_proba(X20)[:, 1]
diag = pd.DataFrame({
    "clause": final["clause"].str[:70],
    "expected": final["expected"],
    "model_p": prob20.round(2),
    "rules_hit": [", ".join(n for n, pat, _ in RULES if re.search(pat, t.lower())) for t in final["clause"]],
})
diag["final"] = ((diag.model_p >= 0.5) | (diag.rules_hit != "")).astype(int)
print(diag[diag.final != diag.expected].to_string(index=False))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

                                                                clause  expected  model_p                rules_hit  final
The landlord will fix any heating problem within 48 hours of being tol         0     0.83 Too little time to leave      1
Landlord must give 30 days notice before any change to the house rules         0     0.52                               1
Tenant may break the lease without penalty if called to active militar         0     0.63                               1
4. ACCESS: The property manager keeps a key and may let himself in day         1     0.47                               0
 The tenant agrees to forfeit the deposit if the lease is not renewed.         1     0.45                               0
Resident shall pay a $50 fee every time a maintenance request is made.         1     0.38                               0
           The renter may not join or organize any tenant association.         1     0.23                               0
If rent is three days la

In [42]:
TEST16 = next(ast.literal_eval(n.value) for n in tree.body
              if isinstance(n, ast.Assign) and getattr(n.targets[0], "id", "") == "TEST")
t16 = pd.DataFrame(TEST16, columns=["text", "clause_type", "risky"])

type_all = pd.read_csv(DATA / "combined_types.csv")
ttr = type_all[type_all.split == "train"].reset_index(drop=True)
Xt = embed(ttr["text"].apply(strip_prefix))
yt = ttr["clause_type"].values
is_seed = (ttr["source"] == "seed").values

def fit_type(mask, seed_w=1.0):
    w = np.where(is_seed[mask], seed_w, 1.0)
    return LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42).fit(Xt[mask], yt[mask], sample_weight=w)

all_rows = np.ones(len(ttr), bool)
type_models = {
    "seed only (64)":            fit_type(is_seed),
    "all 4 datasets":            fit_type(all_rows),
    "all 4 + seed weight 10":    fit_type(all_rows, seed_w=10),
}

type_tests = {
    "Lease TEST (16)": t16,
    "CUAD test":       type_all[(type_all.split == "test") & (type_all.source == "cuad")],
    "LEDGAR test":     type_all[(type_all.split == "test") & (type_all.source == "ledgar")],
}

from sklearn.metrics import f1_score
rows = []
for tname, d in type_tests.items():
    X = embed(d["text"].astype(str).apply(strip_prefix))
    for mname, m in type_models.items():
        p = m.predict(X)
        rows.append((tname, mname, len(d), accuracy_score(d["clause_type"], p),
                     f1_score(d["clause_type"], p, average="macro", zero_division=0)))
type_results = pd.DataFrame(rows, columns=["test set", "model", "n", "accuracy", "macro F1"]).round(2)
print(type_results.to_string(index=False))

# errors on the lease test for the weighted model
m = type_models["all 4 + seed weight 10"]
p16 = m.predict(embed(t16["text"].apply(strip_prefix)))
print("\nLease TEST errors (all 4 + seed weight 10):")
for t, y, p in zip(t16["text"], t16["clause_type"], p16):
    if y != p:
        print(f"  true={y:12s} pred={p:12s} {t}")

type_results.to_csv(PROJECT / "type_results_v2.csv", index=False)

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

       test set                  model   n  accuracy  macro F1
Lease TEST (16)         seed only (64)  16      0.94      0.93
Lease TEST (16)         all 4 datasets  16      0.88      0.87
Lease TEST (16) all 4 + seed weight 10  16      1.00      1.00
      CUAD test         seed only (64) 142      0.74      0.34
      CUAD test         all 4 datasets 142      0.89      0.66
      CUAD test all 4 + seed weight 10 142      0.89      0.53
    LEDGAR test         seed only (64) 103      0.69      0.33
    LEDGAR test         all 4 datasets 103      0.91      0.91
    LEDGAR test all 4 + seed weight 10 103      0.91      0.91


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Lease TEST errors (all 4 + seed weight 10):


In [44]:
type_model = type_models["all 4 + seed weight 10"]
joblib.dump(type_model, PROJECT / "lease_type_model_v2.joblib")
print("saved lease_type_model_v2.joblib")

saved lease_type_model_v2.joblib


In [54]:
%%writefile /Users/marufi/lease-checker/lease_checker_v2.py
import re, json, joblib
from pathlib import Path
import pdfplumber
from sentence_transformers import SentenceTransformer

HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

encoder    = SentenceTransformer("all-MiniLM-L6-v2")
risk_model = joblib.load(HERE / "lease_emb_model_v2.joblib")
type_model = joblib.load(HERE / "lease_type_model_v2.joblib")
RULES      = json.load(open(HERE / "lease_rules.json"))

TYPE_LABEL = {"rent": "Rent", "deposit": "Deposit", "entry": "Landlord entry", "repairs": "Repairs",
              "termination": "Ending the lease", "fees": "Fees & penalties",
              "liability": "Liability & rights", "pets_guests": "Pets & guests"}

def read_file(path):
    path = str(path)
    if path.lower().endswith(".docx"):
        import docx
        return "\n".join(p.text for p in docx.Document(path).paragraphs)
    with pdfplumber.open(path) as pdf:
        return "\n".join((p.extract_text() or "") for p in pdf.pages)

def split_clauses(text):
    # split before "1. ", "2) ", "Section 3", "ARTICLE IV" at a newline OR mid-line after a space
    pattern = r'(?:\n|(?<=[.;:!?])\s)(?=\s*(?:\d{1,2}[\.\)]\s+[A-Z]|Section\s+\d|SECTION\s+\d|ARTICLE\s+\w))'
    parts = re.split(pattern, text)
    if len(parts) < 3:
        parts = re.split(r'\n\s*\n', text)
    return [c for c in (" ".join(p.split()) for p in parts) if len(c.split()) >= 4]
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)

PREFIX = r"^\s*(?:\d{1,2}(?:\.\d+)*[\.\)]?\s+)?(?:[A-Z][A-Z\s/&'\-]{2,}[:\.]\s*)?"
def strip_prefix(t):
    return re.sub(PREFIX, '', t).strip() or t

def check_clause(text, threshold=0.5):
    x = encoder.encode([strip_prefix(text)], normalize_embeddings=True)
    p = float(risk_model.predict_proba(x)[0, 1])
    ctype = TYPE_LABEL.get(type_model.predict(x)[0], "Other")
    reasons = [(n, e) for n, pat, e in RULES if re.search(pat, text.lower())]
    return (p >= threshold or bool(reasons)), round(p, 2), ctype, reasons

def report(parts, title):
    print("=" * 80)
    print(f"REPORT: {title}   |   Clauses checked: {len(parts)}")
    print("=" * 80)
    flagged = 0
    for c in parts:
        risky, p, ctype, reasons = check_clause(c)
        if risky:
            flagged += 1
            print(f'\n⚠ #{flagged} [{ctype}] (model {p})  "{c[:300]}"')
            for n, e in reasons:
                print(f"   → {n}: {e}")
            if not reasons:
                print("   → The meaning is similar to unfair clauses. Read it carefully.")
    print(f"\nFlagged {flagged} of {len(parts)}." if flagged else "\n✓ No risky clauses found.")
    print("Note: This gives warnings, not legal advice.")

def check_document(path, doc_type="lease"):
    parts = split_clauses(read_file(path))
    if doc_type == "notice":
        parts = [s for p in parts for s in split_sentences(p) if len(s) > 40]
    report(parts, Path(path).name)

def check_text(text, title="Pasted text"):
    report(split_clauses("\n" + text), title)

print("Lease checker v2 ready (embeddings + 4 datasets + rules).")


Overwriting /Users/marufi/lease-checker/lease_checker_v2.py


In [56]:
%run /Users/marufi/lease-checker/lease_checker_v2.py
check_document("/Users/marufi/lease-checker/data/leases/Simple-1-Page-Residential-Lease-Agreement.pdf", "lease")
check_text("1. Landlord may enter at any time. 2. Rent is due on the first of each month.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Lease checker v2 ready (embeddings + 4 datasets + rules).
REPORT: Simple-1-Page-Residential-Lease-Agreement.pdf   |   Clauses checked: 6

✓ No risky clauses found.
Note: This gives warnings, not legal advice.
REPORT: Pasted text   |   Clauses checked: 2

⚠ #1 [Landlord entry] (model 0.61)  "1. Landlord may enter at any time."
   → Entry without notice: The landlord can come into your home without warning. Most places require reasonable notice (often 24 hours).

Flagged 1 of 2.
Note: This gives warnings, not legal advice.


In [58]:
check_document("/Users/marufi/lease-checker/data/notices/PRINTABLE-EVICTION-NOTICE-LETTER.pdf", "notice")

REPORT: PRINTABLE-EVICTION-NOTICE-LETTER.pdf   |   Clauses checked: 6

⚠ #1 [Ending the lease] (model 0.54)  "As stated in the lease, violating this agreement is grounds for eviction."
   → The meaning is similar to unfair clauses. Read it carefully.

⚠ #2 [Ending the lease] (model 0.5)  "Failure to do so will result in legal recourse & formal eviction proceedings in accordance with the state law."
   → The meaning is similar to unfair clauses. Read it carefully.

Flagged 2 of 6.
Note: This gives warnings, not legal advice.


In [60]:
%%writefile -a /Users/marufi/lease-checker/lease_checker_v2.py

# ---------- Notice completeness check ----------
NOTICE_CHECKLIST = {
    "reason for eviction":        r"\b(because|reason|due to|breach|violat\w*|non-?payment|unpaid)\b",
    "amount owed (if rent)":      r"\$\s?\d",
    "specific deadline / date":   r"\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|january|february|march|april|may|june|july|august|september|october|november|december|within \d+ days)\b",
    "mention of court / process": r"\b(court|tribunal|hearing|legal proceedings|eviction proceedings|judge|state law)\b",
    "tenant's right to respond":  r"\b(right to (respond|contest|dispute|cure)|you may (pay|dispute|contest)|legal (aid|advice))\b",
}

def notice_completeness(text):
    print("\nCompleteness check (what a fair notice usually contains):")
    for item, pat in NOTICE_CHECKLIST.items():
        ok = re.search(pat, text, flags=re.I)
        print(f"   {'✓' if ok else '✗ missing:'} {item}")

def check_document(path, doc_type="lease"):
    text = read_file(path)
    parts = split_clauses(text)
    if doc_type == "notice":
        parts = [s for p in parts for s in split_sentences(p) if len(s.split()) >= 4]
    report(parts, Path(path).name)
    if doc_type == "notice":
        notice_completeness(text)

Appending to /Users/marufi/lease-checker/lease_checker_v2.py


In [62]:
%run /Users/marufi/lease-checker/lease_checker_v2.py
check_document("/Users/marufi/lease-checker/data/notices/PRINTABLE-EVICTION-NOTICE-LETTER.pdf", "notice")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Lease checker v2 ready (embeddings + 4 datasets + rules).
REPORT: PRINTABLE-EVICTION-NOTICE-LETTER.pdf   |   Clauses checked: 7

⚠ #1 [Ending the lease] (model 0.54)  "As stated in the lease, violating this agreement is grounds for eviction."
   → The meaning is similar to unfair clauses. Read it carefully.

⚠ #2 [Ending the lease] (model 0.5)  "Failure to do so will result in legal recourse & formal eviction proceedings in accordance with the state law."
   → The meaning is similar to unfair clauses. Read it carefully.

Flagged 2 of 7.
Note: This gives warnings, not legal advice.

Completeness check (what a fair notice usually contains):
   ✓ reason for eviction
   ✗ missing: amount owed (if rent)
   ✗ missing: specific deadline / date
   ✓ mention of court / process
   ✗ missing: tenant's right to respond


In [64]:
import shutil
from pathlib import Path
PROJECT = Path.home() / "lease-checker"
GH = Path.home() / "lease-checker-github"

for f in ["lease_checker_v2.py", "lease_emb_model_v2.joblib", "lease_type_model_v2.joblib",
          "final_results_v2.csv", "type_results_v2.csv"]:
    shutil.copy(PROJECT / f, GH / f); print("copied", f)
shutil.copy(PROJECT / "data" / "seed_clauses.csv", GH / "data" / "seed_clauses.csv")

(GH / ".gitignore").write_text("data/external/\n.ipynb_checkpoints/\n__pycache__/\n.DS_Store\n")
req = (GH / "requirements.txt").read_text()
if "datasets" not in req:
    (GH / "requirements.txt").write_text(req.rstrip() + "\ndatasets==3.6.0\n")
print(open(GH / "requirements.txt").read())
print(sorted(p.name for p in GH.iterdir()))

copied lease_checker_v2.py
copied lease_emb_model_v2.joblib
copied lease_type_model_v2.joblib
copied final_results_v2.csv
copied type_results_v2.csv
pandas
scikit-learn
pdfplumber
python-docx
sentence-transformers
joblib
matplotlib
datasets==3.6.0

['.gitignore', '.ipynb_checkpoints', 'README.md', 'data', 'figures', 'final_results_v2.csv', 'final_test_results.csv', 'lease_checker.py', 'lease_checker_v2.py', 'lease_emb_model.joblib', 'lease_emb_model_v2.joblib', 'lease_rules.json', 'lease_type_model_v2.joblib', 'notebooks', 'requirements.txt', 'type_results_v2.csv']


In [66]:
%%writefile /Users/marufi/lease-checker-github/README.md
# Plain-Language Lease & Eviction Notice Checker

A text mining project that reads rental agreements and eviction notices, flags clauses that may be unfair to tenants, labels each clause by topic, and explains why in plain language.

## How it works
1. **Extract** text from PDF or Word files (pdfplumber, python-docx)
2. **Split** documents into clauses (numbered sections or sentences)
3. **Classify risk**: sentence embeddings (all-MiniLM-L6-v2) + Logistic Regression
4. **Classify clause type**: rent, deposit, entry, repairs, termination, fees, liability, pets/guests
5. **Rules**: 11 warning-phrase patterns, each with a plain-language explanation
6. A clause is flagged if the model **or** a rule marks it risky
7. **Eviction notices** also get a completeness check (reason, amount, deadline, court process, right to respond)

## Datasets
| Dataset | What it contains | How it is used |
|---|---|---|
| Own data | 163 lease/notice clauses (131 normal, 32 risky) + 64 seed clauses with types | core training data (weighted ×10) |
| [UNFAIR-ToS](https://huggingface.co/datasets/coastalcph/lex_glue) (LexGLUE) | consumer terms of service; experts marked unfair clauses | **only source of real fair/unfair labels**: 641 risky training clauses (liability, termination, unilateral change, arbitration, jurisdiction) |
| [Lease Contract Review](https://gitlab.com/spyretta.leiv/lease_contract_review) | 23,450 sentences from commercial leases with "red flag" topic tags | clause types + 500 real-lease normal sentences + a 2,000-sentence **false-alarm test** |
| [LEDGAR](https://huggingface.co/datasets/coastalcph/lex_glue) (LexGLUE) | 80,000 contract provisions, 100 labels | clause types (fees, liability, termination) + boilerplate as normal examples |
| [CUAD](https://github.com/TheAtticusProject/cuad) | 510 contracts, 41 clause categories | clause types (fees, liability, termination) |

Lease Contract Review, LEDGAR and CUAD label **topics**, not fairness (e.g. "landlord_repairs" also tags fair repair clauses), so they are not used as risky examples.

**Final training data:** risk model 673 risky / 2,437 normal (before: 32 / 131); clause-type model ~2,000 examples.

## Results: risk detection (full system = model OR rules)
| Test set | n | Old model (own data only) | New v1 (unweighted) | **New v2 (final)** |
|---|---|---|---|---|
| Fresh final lease test (accuracy) | 20 | **0.65** | 0.55 | 0.55 |
| UNFAIR-ToS test (F1, risky) | 368 | 0.30 | 0.74 | **0.71** |
| Real-lease false-alarm rate (lower is better) | 1,999 | 8% | 15% | **7%** |
| LEDGAR boilerplate false-alarm rate | 89 | 1% | 1% | 2% |

v2 settings (own clauses weighted ×10, +500 real-lease normal sentences) were chosen by 5-fold cross-validation on own data and a separate false-alarm validation set, **not** on the final test.

## Results: clause type (accuracy)
| Test set | n | Seed only | **All 4 datasets + seed ×10** |
|---|---|---|---|
| Lease test | 16 | 0.94 | **1.00** |
| CUAD test (unseen contracts) | 142 | 0.74 | **0.89** |
| LEDGAR test | 103 | 0.69 | **0.91** |

## Key findings
- **Domain shift:** 641 terms-of-service unfair clauses raised UNFAIR-ToS F1 from 0.30 to 0.71, but did not help on lease-only risks. All 9 errors on the final lease test are risk types no dataset covers (tenant-association bans, seizing belongings, rent +20% every 3 months, fees per maintenance request).
- **Drowning:** without weighting own clauses, false alarms on real leases doubled (8% → 15%). Weighting ×10 and adding real-lease normal sentences brought it back to 7%.
- **Over-correction on termination:** the model flags fair eviction language (e.g. "formal eviction proceedings in accordance with state law", score 0.50–0.54), because terms of service taught it that termination wording is often unfair.
- **Shortcut learning:** the first model learned writing style instead of meaning; fixed by balancing styles and stripping numbering. UNFAIR-ToS adds the same risk (all lowercase/tokenized), handled by detokenizing.
- **Most legal datasets don't label fairness:** only 1 of 4 labels whether a clause is fair to the weaker party.

## Usage
```python
%run lease_checker_v2.py
check_document("my_lease.pdf", "lease")        # PDF or .docx
check_document("notice.pdf", "notice")         # eviction notice + completeness check
check_text("1. Landlord may enter at any time.")

Overwriting /Users/marufi/lease-checker-github/README.md
